### Draw main graph

In [1]:
import numpy as np
import tensorflow.compat.v1 as tf
from config import *
from GPT_Model import *
from data_pipeline import *

tf.reset_default_graph()

tf.compat.v1.disable_eager_execution()
X = tf.placeholder(tf.int32, [None, hparams.n_time])
Y = tf.placeholder(tf.int32, [None, hparams.n_time])


logits = model(hparams, X)['logits']
cross_entropy = tf.nn.sparse_softmax_cross_entropy_with_logits(labels=Y, logits=logits)
loss = tf.reduce_mean(cross_entropy)

'''
Train
'''
global_step = tf.Variable(0, name='global_step')
learning_rate = tf.train.exponential_decay(1e-3, global_step, 100, 0.7, staircase=False)

if mode == "pretrain":
    train_step = tf.train.AdamOptimizer(learning_rate).minimize(loss, global_step)
elif mode == "finetune":
    optimizer = tf.train.AdamOptimizer(learning_rate)
    output_vars = tf.get_collection(tf.GraphKeys.TRAINABLE_VARIABLES, scope='linear1|linear2')
    train_step = optimizer.minimize(loss, var_list=output_vars, global_step=global_step)

'''
Session Open
'''

# GPU number to use
gpu_options = tf.GPUOptions(visible_device_list="0")
sess = tf.Session(config=tf.ConfigProto(gpu_options=gpu_options))

sess.run(tf.global_variables_initializer())

print('graph create')

Instructions for updating:
If using Keras pass *_constraint arguments to layers.
graph create


### Load model if exist && TensorboardX Logger

In [2]:
import tf_slim as slim
from tensorflow.python import pywrap_tensorflow

load_dir = '../save_model' 
save_dir = '../save_model'


if mode == "pretrain":
    saver = tf.train.Saver()
elif mode == "finetune":
    #只恢复transformer部分的参数
    ref_vars = tf.get_collection(tf.GraphKeys.TRAINABLE_VARIABLES, scope='transformer')
    saver = tf.train.Saver(ref_vars)

restore_file = tf.train.latest_checkpoint(load_dir)
print(restore_file)
if restore_file is not None:
    saver.restore(sess, restore_file)
    print("Model restored.", restore_file)
else:
    print('model not exist.')

#Logger
from tensorboardX import SummaryWriter

class Logger(SummaryWriter):
    def __init__(self, logdir):
        super(Logger, self).__init__(logdir)

    def log(self, log_string, value, iteration):
            self.add_scalar(log_string, value, iteration)
            
logger = Logger(save_dir)  
        

../save_model/checkpoint-1600
INFO:tensorflow:Restoring parameters from ../save_model/checkpoint-1600
Model restored. ../save_model/checkpoint-1600


### Train

In [3]:
from IPython.display import clear_output
from tqdm import tqdm_notebook as tqdm
import matplotlib.pyplot as plt
from time import sleep
import time
import math

print('iteration\t', 'loss\t', 'train_perplexity\t')
while(True):
    for _ in range(100):
        _inputs = []
        _targets = []
        for _ in range(batch_size):
            while(True):
                x, y = get_data(hparams.n_time, data_train_files,'train', 0, type, EventDim)
                if(x.shape == y.shape):
                    break
                 
            _inputs.append(x)
            _targets.append(y)
        _inputs = np.stack(_inputs)
        _targets = np.stack(_targets)
#         print(_inputs.shape, _targets.shape)
        
        _, _global_step, _loss = sess.run([train_step, global_step, loss], 
                                          feed_dict={X: _inputs, 
                                                     Y: _targets})
        
        train_perplexity = math.exp(_loss) #log perplexity和交叉熵等价
        
        if _global_step % 10 == 0:
            logger.log('loss', _loss, _global_step)
            print(str(_global_step)+'\t', str(_loss)+'\t', str(train_perplexity)+'\t')
        
        if _global_step % 100 == 0:
            save_path = saver.save(sess, save_dir + '/checkpoint', global_step=_global_step)
            print("Model saved in path: %s" % save_path)

iteration	 loss	 train_perplexity	
0	 5.8467283	 346.10019874757995	
Model saved in path: ../save_model/checkpoint-0
10	 4.476589	 87.93423476592244	
20	 4.4109173	 82.34496243243437	
30	 4.4219475	 83.258271330632	
40	 4.4364996	 84.47871385958635	
50	 4.4042797	 81.80020165799333	
60	 4.4020534	 81.61828813582444	
70	 4.358279	 78.1225875492704	
80	 4.336117	 76.41024551428305	
90	 4.236256	 69.14848317531373	
100	 3.8294013	 46.034966758788585	
Model saved in path: ../save_model/checkpoint-100
110	 3.7616313	 43.01854278851246	
120	 3.7439325	 42.26386584237664	
130	 3.6841881	 39.81278641953325	
140	 3.6688094	 39.205200895835304	
150	 3.6929646	 40.16373820450373	
160	 3.6686726	 39.19983594242767	
170	 3.6814935	 39.7056510239831	
180	 3.7235003	 41.40908295309583	
190	 3.6595318	 38.84315348229046	
200	 3.734413	 41.86344067910173	
Model saved in path: ../save_model/checkpoint-200
210	 3.6681738	 39.180289050012995	
220	 3.7136834	 41.00456355687679	
230	 3.767472	 43.2705397747

KeyboardInterrupt: 

### Compute perplexity on test set

In [4]:
import math

inputs = []
targets = []

l = len(data_test_files)

for i in range(l): 
    while(True):
        x_test, y_test = get_data(hparams.n_time, data_test_files, 'test', i, 'music', EventDim)
        if(x_test.shape == y_test.shape):
            break       
    inputs.append(x_test)
    targets.append(y_test)
    
inputs = np.stack(inputs)
targets = np.stack(targets)

test_loss = sess.run(loss, feed_dict={X: inputs, Y: targets})
test_perplexity = math.exp(test_loss)
test_perplexity

34.65439993354085